# 15 - Qdrant Hands-on
---

In the previous notebook, we learned FAISS.

FAISS provides fast vector indexing and similarity search.

But production AI systems require much more than fast search.

They need:

- Collections
- Metadata
- Filtering
- Persistence
- CRUD operations
- APIs

This is where **Qdrant** comes in.

Qdrant is a production-ready vector database built for AI applications.

##  History

Libraries such as FAISS provide excellent vector search.

However,

production systems need additional capabilities.

Examples:

- Store metadata
- Filter search results
- Update vectors
- Delete vectors
- Persist data
- Serve multiple applications

Researchers and engineers developed vector databases such as Qdrant to provide these features.

## Think Like a Researcher

Imagine you have one million document embeddings.

FAISS can quickly find similar vectors.

But now a user asks:

"Show only NVIDIA reports from 2024."

Vector similarity alone is not enough.

We also need metadata filtering.

This is exactly why vector databases exist.

In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams
from qdrant_client.models import Distance

In [2]:
client = QdrantClient(":memory:")

In [3]:
client.create_collection(
    collection_name="documents",
    vectors_config=VectorParams(
        size=384,
        distance=Distance.COSINE
    )
)

True

## Observation

A collection is similar to a table in SQL.

Instead of rows,

it stores vectors.

In [5]:
#Upsert
import numpy as np

from qdrant_client.models import PointStruct

points=[]

for i in range(5):

    points.append(

        PointStruct(

            id=i,

            vector=np.random.random(
                384
            ).tolist(),

            payload={

                "company":"NVIDIA",

                "year":2024

            }

        )

    )

client.upsert(

    collection_name="documents",

    points=points

)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [6]:
query = np.random.random(
    384
).tolist()

results = client.query_points(

    collection_name="documents",

    query=query,

    limit=3

)

results.points

[ScoredPoint(id=0, version=0, score=0.7807237996444248, payload={'company': 'NVIDIA', 'year': 2024}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=2, version=0, score=0.7747963408875717, payload={'company': 'NVIDIA', 'year': 2024}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=3, version=0, score=0.7589748736659503, payload={'company': 'NVIDIA', 'year': 2024}, vector=None, shard_key=None, order_value=None)]

## Payload

Payload is metadata.

Example

Company

↓

NVIDIA

Year

↓

2024

Quarter

↓

Q1

Payload allows filtering during search.

### Metadata Filtering

In [7]:
from qdrant_client.models import Filter

from qdrant_client.models import FieldCondition

from qdrant_client.models import MatchValue

results = client.query_points(

    collection_name="documents",

    query=query,

    query_filter=Filter(

        must=[

            FieldCondition(

                key="company",

                match=MatchValue(

                    value="NVIDIA"

                )

            )

        ]

    ),

    limit=3

)

results.points

[ScoredPoint(id=0, version=0, score=0.7807237996444248, payload={'company': 'NVIDIA', 'year': 2024}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=2, version=0, score=0.7747963408875717, payload={'company': 'NVIDIA', 'year': 2024}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=3, version=0, score=0.7589748736659503, payload={'company': 'NVIDIA', 'year': 2024}, vector=None, shard_key=None, order_value=None)]

## Think

Without metadata,

the database searches

every document.

With metadata,

the search is limited to

only relevant documents.

This improves precision.

In [8]:
client.upsert(

    collection_name="documents",

    points=[

        PointStruct(

            id=0,

            vector=np.random.random(
                384
            ).tolist(),

            payload={

                "company":"Apple",

                "year":2025

            }

        )

    ]

)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [9]:
from qdrant_client.models import PointIdsList

client.delete(

    collection_name="documents",

    points_selector=PointIdsList(

        points=[4]

    )

)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

## Qdrant Search Pipeline

```
Documents

↓

Embeddings

↓

Collection

↓

Payload

↓

User Query

↓

Embedding

↓

Metadata Filter

↓

Similarity Search

↓

Top K Results
```

## Comparison

| Feature | FAISS | Qdrant |
|----------|-------|---------|
| Vector Search | ✅ | ✅ |
| Metadata Filtering | ❌ | ✅ |
| Collections | ❌ | ✅ |
| CRUD | Limited | ✅ |
| REST API | ❌ | ✅ |
| Persistence | Basic | ✅ |
| Production Database | ❌ | ✅ |

##  Applications

Qdrant is used in

- RAG

- Semantic Search

- AI Assistants

- Enterprise Search

- Recommendation Systems

- Financial AI

##  Think Like a Researcher

Now we have another question.

Should we rely only on vector search?

Sometimes,

keyword search is also important.

For example,

searching for

"NVIDIA"

should return documents containing exactly that company name,

even if the embedding is not very similar.

Researchers combine

Keyword Search

+

Vector Search.

This approach is called

**Hybrid Search**.